# לקח 05 - RAG אייג'נטי


## הגדרה

מחברת זו מדגימה את תבנית Agentic RAG (יצירה משופרת על ידי אחזור) באמצעות Microsoft Agent Framework.

**דרישות מוקדמות:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — נקודת הקצה של שירות Azure AI Search שלך
- `AZURE_SEARCH_API_KEY` — מפתח ה-API של Azure AI Search שלך
- פריסת Azure OpenAI מוגדרת דרך משתני סביבה
- Azure CLI מאומת (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## מה זה Agentic RAG?

RAG המסורתי פועל לפי צינור עבודה קבוע: משיבה מסמכים, ואז מייצר תגובה. **Agentic RAG** הולך רחוק יותר בכך שנתן לסוכן אוטונומיה להחליט **מתי** ו**איך** לאסוף מידע.

עם Agentic RAG, הסוכן יכול:
- **להחליט** אם יש צורך במשיכת מידע לפני מענה על שאלה
- **לבחור** באיזו מקור נתונים או כלי לשאול
- **להעריך** את התוצאות שהתקבלו ולבצע משיכות נוספות אם הניסיון הראשון לא מספיק
- **לשלב** מידע משלבי משיכה מרובים לתשובה עקבית

זה הופך את הסוכן ליותר גמיש ומדויק בהשוואה לצינור עבודה סטטי של משיכה-ואז-יצירה.


## יצירת כלי חיפוש

ב-Agentic RAG, מקורות נתונים חיצוניים הם עטופים כ**כלים** שהסוכן יכול להפעיל על פי דרישה. זה מאפשר לסוכן להתייחס לאחזור רק כפעולה נוספת שהוא יכול לבצע, ולא כשלב חובה.

להלן אנו מגדירים בסיס ידע טיולים וחשופים אותו ככלי שהסוכן יכול לקרוא לו כדי לחפש מידע על יעדים.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## בניית סוכן ה-RAG

כעת אנו יוצרים סוכן שמופקד על **לשלוף מידע תמיד לפני מתן תשובה**. הסוכן משתמש בכלי `search_travel_knowledge` כדי לבסס את תגובותיו על מאגר הידע במקום להסתמך על נתוני האימון שלו.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## אחזור איטרטיבי — תבנית יוצר-בודק

יתרון מרכזי של Agentic RAG הוא **אחזור איטרטיבי**. הסוכן יכול לבצע מספר סבבי חיפוש כדי לאמת, לחדד או להרחיב על ממצאיו הראשוניים — בדומה לזרימת עבודה של "יוצר-בודק":

1. **שלב היוצר**: הסוכן מאחזר מידע ראשוני ומנסח תגובה.
2. **שלב הבודק**: הסוכן מבצע אחזור נוסף כדי לאמת פרטים או למלא פערים.

להלן, נשאל את הסוכן שאלה שדורשת השוואה בין מספר יעדים, מה שגורם לו לחפש מספר פעמים.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## סיכום

בשיעור זה למדת כיצד לבנות מערכת **Agentic RAG** באמצעות מסגרת הסוכן של מיקרוסופט:

- **Agentic RAG** מאפשרת לסוכנים להחליט באופן עצמאי מתי לאחזר מידע, מה שהופך את האחזור לדינמי במקום קבוע.
- **כלים כמקורות נתונים**: מאגרי ידע חיצוניים (כגון Azure AI Search) עטופים ככלים שהסוכן יכול להפעיל.
- **אחזור איטרטיבי**: תבנית ה-maker-checker מאפשרת לסוכן לבצע מספר סבבי אחזור — חיפוש, אימות, ושיפור — לפני הפקת תשובה סופית.

בייצור, תחליף את `TRAVEL_KNOWLEDGE_BASE` הזמני באינדקס אמיתי של Azure AI Search כדי לטפל באחזור מסמכי נסיעות בקנה מידה גדול.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור אחריות**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). בעוד שאנו שואפים לדיוק, יש לשים לב כי תרגומים ממוחשבים עשויים להכיל שגיאות או אי-דיוקים. המסמך המקורי בשפת המקור שלו נחשב למקור הסמכותי. למידע קריטי מומלץ תרגום מקצועי ידי אדם. אנו לא אחראים לכל אי-הבנות או פירושים שגויים הנובעים משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
